# MODUL PRAKTIKUM BIG DATA
## Pemrosesan Terdistribusi PySpark RDD di Databricks Community Edition

**Nama:** Alya Maura Raditha  
**NIM:** 52240003

Memproses data besar (*Big Data*) menggunakan **PySpark RDD (Resilient Distributed Dataset)** di platform cloud **Databricks Community Edition**.

---

### 📌 Dataset yang Digunakan
* **Nama Dataset:** UCI / Kaggle – *Online Retail II Dataset*
* **Deskripsi:** Dataset transaksi nyata e-commerce ritel sebanyak lebih dari 500.000 baris record transaksi.
* **Tautan Dataset:**
  * **Kaggle:** [UCI Online Retail II Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail)
  * **UCI Direct Link:** [UCI Machine Learning Repository Zip/CSV](https://archive.ics.uci.edu/ml/machine-learning-databases/00352/)

### 📝 Catatan Adaptasi ke Google Colab
Sesuai ketentuan tugas (*tidak menyimpan/memproses data di komputer lokal*), notebook ini dijalankan di **Google Colab**. Bagian yang bersifat khusus Databricks disesuaikan sebagai berikut:

| Komponen pada modul asli (Databricks) | Penyesuaian pada notebook ini (Colab) |
|---|---|
| `%sh wget ...` dan `cp ... /dbfs/FileStore/tables/` | Dataset dibaca dari Google Drive (`/content/drive/My Drive/data/OnlineRetail.csv`) |
| Variabel `spark` dan `sc` tersedia otomatis | `SparkSession` dibuat manual (mode `local[*]`) |
| Path `dbfs:/FileStore/tables/...` | Path Colab `/content/data/...` |

**Penting – Menjalankan notebook:** gunakan menu **Runtime → Run all**. Seluruh sel saling bergantung dan harus dijalankan berurutan. Hasil `print()` setiap tahapan direkam otomatis dan disimpan ke file **`/content/output_1.txt`** pada sel paling akhir.

### 🔎 Catatan tentang Interpretasi
Setiap sel analisis diikuti sel **Interpretasi Hasil** yang ditulis berdasarkan hasil eksekusi nyata pada dataset `OnlineRetail.csv` (541,909 baris) di Google Colab (2 core). Angka non-waktu (jumlah, nilai, persentase, peringkat) bersifat deterministik dan akan sama saat dijalankan ulang. **Angka waktu eksekusi** (detik) bergantung pada kondisi runtime Colab dan dapat berbeda setiap kali dijalankan, jadi yang perlu dibaca adalah **polanya**, bukan angka mutlaknya.

Seluruh analisis mengikuti instruksi tugas apa adanya. Bila interpretasi menyebut adanya distorsi pada data (mis. pesanan yang kemudian dibatalkan), itu disajikan sebagai **catatan analitis**; kode tidak diubah dari ketentuan tugas.

## 0. Persiapan Lingkungan Google Colab

In [ ]:
# ==============================================================================
# 0. PERSIAPAN LINGKUNGAN GOOGLE COLAB
# ==============================================================================
# Instal PySpark. Versi 3.5.3 dipin agar perilaku fungsi (mis. to_timestamp, cast)
# konsisten di seluruh runtime Colab.
!pip install -q pyspark==3.5.3 openpyxl

import os, sys, io, shutil, subprocess, time, datetime

# Spark membutuhkan Java. Colab umumnya sudah menyediakannya; jika belum, pasang otomatis.
if shutil.which("java") is None:
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=True)

print("Java tersedia :", shutil.which("java") is not None)
print("Versi Python  :", sys.version.split()[0])

In [ ]:
# ------------------------------------------------------------------------------
# Perekam output: setiap hasil print() juga disimpan ke LOG_BUFFER, lalu di sel
# terakhir ditulis ke file output_1.txt (output aktual hasil eksekusi notebook ini).
# ------------------------------------------------------------------------------
class _Tee:
    def __init__(self, stream, buf):
        self._stream, self._buf = stream, buf
    def write(self, s):
        self._buf.write(s)
        return self._stream.write(s)
    def flush(self):
        self._stream.flush()
    def __getattr__(self, name):
        return getattr(self._stream, name)

LOG_BUFFER = io.StringIO()
if not isinstance(sys.stdout, _Tee):
    sys.stdout = _Tee(sys.stdout, LOG_BUFFER)

def banner(title):
    """Cetak judul bagian agar file output tersusun sistematis sesuai urutan tahapan."""
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

RUN_STARTED = datetime.datetime.now()

## 1. Pengambilan Dataset dari Google Drive

Dataset **`OnlineRetail.csv`** dibaca dari Google Drive pada folder **`/content/drive/My Drive/data`**. Saat sel dijalankan, Colab akan meminta izin akses Drive (klik **Connect to Google Drive** dan pilih akun yang menyimpan file tersebut).

File kemudian diseragamkan menjadi CSV UTF-8 pada disk lokal Colab (`/content/data/online_retail.csv`) dengan kolom: `InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country`. Penyeragaman ini menangani *encoding* asli file Kaggle (ISO-8859-1) agar teks tidak rusak.

In [ ]:
import glob
import pandas as pd

banner("1. PENGAMBILAN DATASET DARI GOOGLE DRIVE")

# Mount Google Drive (di luar Colab, langkah ini dilewati dan folder yang sudah ada dipakai apa adanya)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("(Bukan Google Colab: mount Drive dilewati.)")

DRIVE_DIR = "/content/drive/My Drive/data"
DRIVE_FILE = f"{DRIVE_DIR}/OnlineRetail.csv"

if not os.path.exists(DRIVE_FILE):
    isi = sorted(os.listdir(DRIVE_DIR)) if os.path.isdir(DRIVE_DIR) else "(folder tidak ditemukan)"
    raise FileNotFoundError(
        f"File tidak ditemukan: {DRIVE_FILE}\n"
        f"Isi folder {DRIVE_DIR}: {isi}\n"
        "Pastikan Drive sudah terhubung dan nama file/folder persis sama (huruf besar-kecil berpengaruh)."
    )

LOCAL_DIR = "/content/data"
os.makedirs(LOCAL_DIR, exist_ok=True)
CSV_PATH = f"{LOCAL_DIR}/online_retail.csv"       # salinan lokal yang dibaca Spark

# Pemetaan nama kolom agar skema seragam (versi 'Online Retail' dan 'Online Retail II' berbeda penamaan)
COL_ALIASES = {"Invoice": "InvoiceNo", "Customer ID": "CustomerID", "Price": "UnitPrice"}
REQUIRED_COLS = ["InvoiceNo", "StockCode", "Description", "Quantity",
                 "InvoiceDate", "UnitPrice", "CustomerID", "Country"]

def normalize_to_csv(raw_path, csv_path):
    """Membaca file mentah (csv/xlsx), menyeragamkan skema, lalu menyimpan sebagai CSV UTF-8."""
    if raw_path.lower().endswith((".xlsx", ".xls")):
        df = pd.concat(pd.read_excel(raw_path, sheet_name=None, engine="openpyxl").values(), ignore_index=True)
    else:
        try:
            df = pd.read_csv(raw_path, encoding="utf-8")
        except UnicodeDecodeError:
            df = pd.read_csv(raw_path, encoding="ISO-8859-1")   # encoding asli OnlineRetail.csv (Kaggle)
    df = df.rename(columns=COL_ALIASES)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Kolom wajib tidak ditemukan: {missing}. Kolom tersedia: {list(df.columns)}")
    df = df[REQUIRED_COLS].copy()
    df["InvoiceNo"] = df["InvoiceNo"].astype(str)
    df["StockCode"] = df["StockCode"].astype(str)
    df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce").astype("Int64")   # tanpa akhiran ".0"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    return len(df)

n_rows = normalize_to_csv(DRIVE_FILE, CSV_PATH)

print(f"Sumber (Drive)  : {DRIVE_FILE}  ({os.path.getsize(DRIVE_FILE)/1e6:,.2f} MB)")
print(f"Salinan lokal   : {CSV_PATH}  ({os.path.getsize(CSV_PATH)/1e6:,.2f} MB)")
print(f"Jumlah baris    : {n_rows:,}")
print("3 baris pertama file:")
with open(CSV_PATH, encoding="utf-8") as f:
    for _ in range(3):
        print("  ", f.readline().rstrip())

## 2. Inisialisasi & Verifikasi SparkContext

In [ ]:
from pyspark.sql import SparkSession, functions as F

# Di Colab, 'spark' dan 'sc' dibuat manual (di Databricks tersedia otomatis).
# Catatan: sengaja TIDAK mengatur spark.default.parallelism agar jumlah partisi RDD
# sepenuhnya ditentukan oleh .repartition() pada eksperimen Tugas B1.
spark = (SparkSession.builder
         .master("local[*]")
         .appName("PraktikumBigData_PySpark_RDD")
         .config("spark.driver.memory", "4g")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

banner("2. INFORMASI SPARK CLUSTER (COLAB)")
print(f"Versi PySpark    : {spark.version}")
print(f"Master Node      : {sc.master}")
print(f"Nama Aplikasi    : {sc.appName}")
print(f"Default Parallel : {sc.defaultParallelism} core/thread")

## 3. Membaca Dataset dari Colab ke PySpark RDD

In [ ]:
banner("3. MEMBACA DATASET KE PYSPARK RDD")

# Membaca dataset CSV ke PySpark DataFrame lalu mengonversinya ke RDD Tuple
file_path = f"file://{CSV_PATH}"

df_raw = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("escape", '"')          # tanda kutip ganda di dalam teks ditulis sebagai ""
          .load(file_path))

# Pemeriksaan struktur & karakteristik dataset sebelum diproses
print("--- SKEMA DATASET (hasil inferSchema) ---")
df_raw.printSchema()
print(f"Jumlah Total Record Mentah : {df_raw.count():,}")

faltan = [c for c in ["InvoiceNo", "StockCode", "Description", "Quantity",
                      "InvoiceDate", "UnitPrice", "CustomerID", "Country"] if c not in df_raw.columns]
assert not faltan, f"Kolom wajib tidak ditemukan pada dataset: {faltan}"

# Pembersihan data awal: Hapus baris dengan nilai NULL pada kolom vital
df_clean = df_raw.dropna(subset=["InvoiceNo", "CustomerID", "UnitPrice", "Quantity"])

# Konversi DataFrame ke RDD Tuple
# Format Tuple: (InvoiceNo, StockCode, Description, Quantity, UnitPrice, CustomerID, Country)
# CustomerID dinormalisasi menjadi string bilangan bulat (mis. "17850", bukan "17850.0").
rdd_raw = df_clean.rdd.map(lambda row: (
    str(row['InvoiceNo']),
    str(row['StockCode']),
    str(row['Description']) if row['Description'] is not None else 'UNKNOWN',
    int(row['Quantity']),
    float(row['UnitPrice']),
    str(int(float(row['CustomerID']))),
    str(row['Country'])
))

# Optimasi Partisi Data terdistribusi
rdd = rdd_raw.repartition(8)

# RDD sumber utama dicache (MEMORY_AND_DISK) agar tidak membaca & mem-parsing ulang CSV
# pada setiap aksi. Isi RDD tidak diubah sedikit pun.
from pyspark import StorageLevel
rdd.persist(StorageLevel.MEMORY_AND_DISK)

print(f"\nJumlah Total Record Terbersih: {rdd.count():,}")
print(f"Jumlah Partisi RDD            : {rdd.getNumPartitions()}")
print("\nContoh 3 Transaksi Pertama:")
for row in rdd.take(3):
    print(row)

## 4. Transformasi RDD (`filter` & `map`)

In [ ]:
banner("4. TRANSFORMASI RDD (filter & map)")

# Filter transaksi valid: Quantity > 0, UnitPrice > 0, dan berasal dari 'United Kingdom'
rdd_uk_valid = rdd.filter(lambda x: x[3] > 0 and x[4] > 0 and x[6] == 'United Kingdom')

# Transformasi map untuk membentuk Tuple Key-Value: (InvoiceNo, TotalSpent)
# TotalSpent = Quantity * UnitPrice
rdd_invoice_spent = rdd_uk_valid.map(lambda x: (x[0], round(x[3] * x[4], 2)))

print(f"Jumlah Transaksi Valid UK: {rdd_uk_valid.count():,}")
print("Contoh Pair RDD (InvoiceNo, TotalSpent):", rdd_invoice_spent.take(3))

#### 🔎 Interpretasi Hasil — Tahap 3 & 4 (Kualitas Data dan Cakupan Analisis)

**Temuan utama**
- Dari **541,909** record mentah, hanya **406,829 record (75.07%)** yang lolos pembersihan. Sebanyak **135,080 record (24.93%)** dibuang karena `CustomerID` kosong. Seluruh 1,454 baris yang `Description`-nya kosong ternyata juga tanpa `CustomerID`, sehingga ikut terbuang dan tidak ada nilai `UNKNOWN` yang tersisa pada RDD.
- Filter Tahap 4 (negara *United Kingdom*, `Quantity > 0`, `UnitPrice > 0`) menyisakan **354,321 record (87.09% dari data bersih)**. Sebanyak 52,508 record tersingkir, terdiri dari 44,951 transaksi non-UK, 7,533 baris UK dengan `Quantity ≤ 0` (retur/pembatalan), dan 24 baris UK dengan harga 0.

**Makna**
- Satu dari empat baris data tidak memiliki identitas pelanggan (kemungkinan transaksi tanpa akun terdaftar). Konsekuensinya, seluruh analisis berikutnya hanya mencakup transaksi pelanggan terdaftar, sehingga nilai omzet pada notebook ini **bukan total omzet perusahaan**.
- Pasar UK sangat dominan: 361,878 dari 406,829 baris bersih (≈ 89.0%) berasal dari UK, sehingga hasil yang memakai `rdd_uk_valid` mewakili hampir seluruh aktivitas yang terekam.
- Contoh Pair RDD `(InvoiceNo, TotalSpent)` menunjukkan bahwa satu invoice terdiri dari beberapa baris produk; karena itu agregasi per invoice pada tahap berikutnya (`reduceByKey`) diperlukan untuk memperoleh nilai belanja per transaksi.

## 5. Agregasi RDD (`reduceByKey` & Sorting)

In [ ]:
banner("5. AGREGASI RDD (reduceByKey & Sorting)")
import time

# Hitung Total Nilai Belanja per Invoice ID menggunakan reduceByKey
start_time = time.time()

rdd_total_per_invoice = rdd_invoice_spent.reduceByKey(lambda a, b: a + b)

# Mengurutkan transaksi dari nilai belanja tertinggi
rdd_sorted_invoice = rdd_total_per_invoice.sortBy(lambda x: x[1], ascending=False)

top_10_invoices = rdd_sorted_invoice.take(10)
execution_time = time.time() - start_time

print(f"Waktu Eksekusi Agregasi RDD: {execution_time:.4f} detik\n")
print("--- TOP 10 INVOICE DENGAN TRANSAKSI TERBESAR ---")
for invoice, total in top_10_invoices:
    print(f"Invoice ID: {invoice:<10} | Total Belanja: £ {total:,.2f}")

#### 🔎 Interpretasi Hasil — Tahap 5 (Top 10 Invoice Terbesar)

**Temuan utama**
- Terdapat **16,646 invoice UK valid** dengan total nilai **£7,308,391.55**. Rata-rata nilai per invoice **£439.05**, sedangkan mediannya **£299.56**. Rata-rata yang 1.47 kali lebih besar daripada median menunjukkan distribusi yang **condong ke kanan**: mayoritas invoice bernilai kecil-menengah, ditarik naik oleh sejumlah invoice besar.
- Sepuluh invoice teratas bernilai **£433,383.03 (5.93% dari total)**. Invoice terbesar, **581483 (£168,469.60)**, menyumbang 2.31% total, atau sekitar **383.7 kali** rata-rata nilai invoice.

**Catatan kritis: tiga dari sepuluh invoice teratas ternyata dibatalkan**
- **581483** hanya berisi 1 baris: 80,995 unit *PAPER CRAFT , LITTLE BIRDIE* × £2.08 (pelanggan 16446, 9 Desember 2011). Pada data terdapat invoice pembatalan **C581484** dengan jumlah dan nilai yang persis sama.
- **541431** hanya berisi 1 baris: 74,215 unit *MEDIUM CERAMIC TOP STORAGE JAR* × £1.04 (pelanggan 12346, 18 Januari 2011), dibatalkan oleh **C541433** dengan nilai persis sama (£77,183.60).
- **556444** berisi 60 unit *PICNIC BASKET WICKER 60 PIECES* × £649.50 = £38,970.00 (pelanggan 15098, pukul 15:28), dan tiga menit kemudian (15:31) muncul **C556445** berupa satu baris bertanda "Manual" bernilai −£38,970.00.
- Ketiga invoice tersebut bernilai **£299,784.10**, yakni **69.2% dari nilai Top 10** dan 4.10% dari total UK valid. Invoice ke-11 (540815) juga sebagian besar nilainya (£15,160.90) dibatalkan.

**Makna**
- Filter Tahap 4 hanya menyaring `Quantity > 0`; baris pembatalan (berkuantitas negatif) terbuang, tetapi **pesanan aslinya tetap tersisa**. Akibatnya nilai tertinggi pada ranking ini adalah **omzet semu**. Tanpa ketiga invoice tersebut, invoice terbesar berikutnya (567423) bernilai £31,698.16, hanya sekitar 19% dari angka teratas semula.
- Untuk analisis bisnis yang sebenarnya, invoice yang memiliki pasangan pembatalan sebaiknya dikeluarkan atau dihitung neto. Kode tidak diubah karena instruksi tugas hanya meminta filter `Quantity > 0`, `UnitPrice > 0`, dan UK.

**Catatan waktu**
- Waktu eksekusi 4.6381 detik (Colab, 2 core) mencakup **seluruh rangkaian transformasi lazy** (`reduceByKey`, lalu `sortBy` yang memicu pengambilan sampel dan shuffle kedua, lalu `take(10)`), bukan hanya `reduceByKey`. Spark baru menjalankan semuanya saat aksi `take(10)` dipanggil.

## 6. Optimasi RDD: Caching & Persistence

In [ ]:
banner("6. OPTIMASI RDD: CACHING & PERSISTENCE")
from pyspark import StorageLevel

# Simpan RDD yang sering diakses ke dalam RAM Cluster
rdd_sorted_invoice.persist(StorageLevel.MEMORY_ONLY)

# Eksekusi Aksi Pertama (Cache Warmup)
t0 = time.time()
count_1 = rdd_sorted_invoice.count()
t1 = time.time() - t0

# Eksekusi Aksi Kedua (Membaca langsung dari RAM)
t2 = time.time()
count_2 = rdd_sorted_invoice.count()
t3 = time.time() - t2

print(f"Jumlah Invoice (aksi pertama / kedua)        : {count_1:,} / {count_2:,}")
print(f"Eksekusi Aksi Pertama (Tanpa Cache Warmup) : {t1:.4f} detik")
print(f"Eksekusi Aksi Kedua (Menggunakan Cache)     : {t3:.4f} detik")

# Lepaskan memori setelah selesai
_ = rdd_sorted_invoice.unpersist()

#### 🔎 Interpretasi Hasil — Tahap 6 (Caching & Persistence)

**Temuan utama (Colab, 2 core)**
- Aksi pertama (`count()`, tanpa cache) memerlukan **0.9118 detik**; aksi kedua (membaca dari memori) **0.4562 detik**. Aksi kedua **2.00 kali lebih cepat** (waktu berkurang sekitar 50.0%). Kedua aksi menghasilkan jumlah yang sama, **16,646 invoice**, sehingga cache tidak mengubah hasil.

**Makna**
- Pada aksi pertama, Spark menjalankan rantai `reduceByKey` dan `sortBy` (termasuk dua shuffle) lalu menyimpan partisi hasilnya di RAM karena `persist(MEMORY_ONLY)`. Aksi kedua langsung membaca partisi tersimpan tanpa menghitung ulang rantai tersebut.
- Percepatan hanya sekitar 2 kali, bukan jauh lebih besar, karena (a) RDD sumber `rdd` sudah di-cache sehingga CSV tidak perlu dibaca ulang pada aksi pertama; penghematan di sini murni dari menghindari shuffle dan sort; dan (b) aksi kedua masih menanggung biaya tetap: meluncurkan task untuk 8 partisi dan melakukan deserialisasi objek Python.
- Pelajaran praktis: caching paling bermanfaat untuk RDD yang **mahal dihitung dan dipakai berulang** oleh beberapa aksi. Untuk RDD yang hanya dipakai sekali, caching hanya menghabiskan memori tanpa manfaat. Angka detik dapat berbeda pada setiap eksekusi, tetapi aksi kedua seharusnya selalu lebih cepat.

## 7. Analisis Produk Paling Laris

In [ ]:
banner("7. ANALISIS PRODUK PALING LARIS")

# Analisis Top 5 Produk Paling Banyak Terjual (Berdasarkan Total Kuantitas)
# Pair RDD: (Description, Quantity)
rdd_product_qty = rdd_uk_valid.map(lambda x: (x[2], x[3])) \
                              .reduceByKey(lambda a, b: a + b) \
                              .sortBy(lambda x: x[1], ascending=False)

print("--- TOP 5 PRODUK TERLARIS ---")
for product, total_qty in rdd_product_qty.take(5):
    print(f"Produk: {product:<40} | Jumlah Terjual: {total_qty:,} unit")

#### 🔎 Interpretasi Hasil — Tahap 7 (Top 5 Produk Terlaris)

**Temuan utama**
- Peringkat berdasarkan total unit (UK valid): *PAPER CRAFT , LITTLE BIRDIE* (80,995), *MEDIUM CERAMIC TOP STORAGE JAR* (76,919), *WORLD WAR 2 GLIDERS ASSTD DESIGNS* (49,182), *JUMBO BAG RED RETROSPOT* (41,981), dan *WHITE HANGING HEART T-LIGHT HOLDER* (34,648).
- Kelima produk ini hanya **6.67%** dari 4,256,740 unit yang terjual, dari 3,844 nama produk berbeda. Artinya permintaan **tidak terkonsentrasi** pada segelintir produk (*long tail*).

**Catatan kritis: dua peringkat teratas adalah artefak pesanan yang dibatalkan**
- Produk peringkat 1 hanya terdiri dari **satu baris transaksi** (80,995 unit), yaitu pesanan yang dibatalkan melalui C581484 (lihat interpretasi Tahap 5). Produk peringkat 2 terdiri dari baris 74,215 unit (**96.5%** dari totalnya) yang juga dibatalkan.
- Jika retur diperhitungkan (jumlah neto di UK), *PAPER CRAFT , LITTLE BIRDIE* menjadi **0 unit** dan *MEDIUM CERAMIC TOP STORAGE JAR* hanya **2,452 unit**. Lima produk terlaris neto adalah *WORLD WAR 2 GLIDERS ASSTD DESIGNS* (47,982), *JUMBO BAG RED RETROSPOT* (40,880), *ASSORTED COLOUR BIRD ORNAMENT* (32,679), *WHITE HANGING HEART T-LIGHT HOLDER* (32,097), dan *POPCORN HOLDER* (28,562).

**Makna**
- Produk terlaris yang sebenarnya adalah *WORLD WAR 2 GLIDERS* dan *JUMBO BAG RED RETROSPOT*. *WHITE HANGING HEART T-LIGHT HOLDER* menonjol dari sisi frekuensi: ia produk dengan baris transaksi terbanyak (2,070 baris pada data bersih), sehingga populer secara konsisten, bukan karena beberapa pesanan besar.
- Kolom "Jumlah Terjual" adalah **unit**, bukan nilai rupiah/pound; produk berharga murah dapat unggul di unit tetapi tidak di pendapatan. Kode tidak diubah karena instruksi hanya meminta total kuantitas pada `rdd_uk_valid`.

## 8. Tugas Mandiri Mahasiswa

> **Dilarang menyimpan data ataupun memproses data di komputer lokal.**
> Gunakan Google Colab, Databricks, atau Kaggle, atau cloud service yang lain.

**Petunjuk Umum:**
1. Kerjakan tugas-tugas di bawah ini menggunakan transformasi dan aksi PySpark RDD.
2. Jangan mengubah data sumber utama (`rdd` / `rdd_uk_valid`).
3. Tuliskan skrip dan jalankan sel untuk menampilkan output jawaban Anda!

### KATEGORI A: ANALISIS DATA & EKSPLORASI BISNIS (BUSINESS INTELLIGENCE)

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS A1: Analisis Pelanggan Utama Luar Negeri (Non-UK)
# Instruksi:
# 1. Filter dataset untuk mengambil transaksi yang berasal dari LUAR negara 'United Kingdom' (Country != 'United Kingdom').
# 2. Buat Pair RDD dengan Key = CustomerID dan Value = Total Spent (Quantity * UnitPrice).
# 3. Hitung total belanja setiap CustomerID dan tampilkan Top 5 Pelanggan dengan pengeluaran terbesar!
# ------------------------------------------------------------------------------
banner("TUGAS A1 — TOP 5 PELANGGAN LUAR NEGERI (NON-UK)")

# 1. Filter transaksi di luar United Kingdom (sesuai instruksi: hanya filter negara)
rdd_non_uk = rdd.filter(lambda x: x[6] != 'United Kingdom')

# 2. Pair RDD: (CustomerID, Quantity * UnitPrice)
rdd_cust_spent = rdd_non_uk.map(lambda x: (x[5], x[3] * x[4]))

# 3. Total belanja per CustomerID, lalu ambil Top 5 (takeOrdered -> hasil deterministik)
rdd_cust_total = rdd_cust_spent.reduceByKey(lambda a, b: a + b)
top5_customers = rdd_cust_total.takeOrdered(5, key=lambda kv: (-kv[1], kv[0]))

# Informasi pendukung: negara asal tiap pelanggan Top 5
top5_ids = [cid for cid, _ in top5_customers]
cust_country = (rdd_non_uk.filter(lambda x: x[5] in top5_ids)
                          .map(lambda x: (x[5], x[6])).distinct()
                          .groupByKey().mapValues(lambda v: ", ".join(sorted(v)))
                          .collectAsMap())

print(f"Jumlah record transaksi non-UK : {rdd_non_uk.count():,}")
print(f"Jumlah pelanggan non-UK unik   : {rdd_cust_total.count():,}")
print("\n--- TOP 5 PELANGGAN LUAR NEGERI BERDASARKAN TOTAL BELANJA ---")
for rank, (cid, total) in enumerate(top5_customers, start=1):
    print(f"{rank}. CustomerID: {cid:<8} | Negara: {cust_country.get(cid, '-'):<18} | Total Belanja: £ {total:,.2f}")

#### 🔎 Interpretasi Hasil — Tugas A1 (Top 5 Pelanggan Luar Negeri)

**Temuan utama**
- Dari **422 pelanggan non-UK** (44,951 record) dengan total belanja neto **£1,532,192.42**, lima pelanggan teratas menyumbang **£679,472.05 atau 44.35%**. Artinya **1.18% pelanggan menghasilkan hampir separuh nilai transaksi** luar negeri.
- Peringkat: (1) **14646** Netherlands £279,489.02, (2) **14911** EIRE £132,572.62, (3) **12415** Australia £123,725.45, (4) **14156** EIRE £113,384.14, (5) **17404** Sweden £30,300.82. Empat pelanggan teratas masing-masing di atas £113,000 (gabungan 42.37%), lalu nilainya turun tajam ke £30,300.82 pada peringkat 5 (sekitar 3.7 kali lebih kecil dari peringkat 4).
- Pelanggan **14646** saja bernilai **18.24%** dari seluruh belanja non-UK, dengan 77 invoice. Rata-rata belanja per pelanggan £3,630.79, jauh di atas mediannya £1,001.35 (3.63 kali), menandakan distribusi sangat condong ke kanan.

**Makna: pendapatan negara tertentu bergantung pada satu-dua akun**
- **Netherlands** (£284,662): 98.2% berasal dari pelanggan 14646 sendiri. **EIRE** (£250,285) hanya memiliki 3 pelanggan, dan dua teratas (14911 dan 14156) menyumbang 98.3%. **Australia** (£137,077): 9 pelanggan, satu pelanggan (12415) 90.3%. **Sweden** (£36,596): 8 pelanggan, satu pelanggan (17404) 82.8%.
- Sebaliknya **Germany** (£221,698; 95 pelanggan) dan **France** (£196,713; 87 pelanggan) bersifat **berbasis luas**: pelanggan teratas hanya 8.5% dan 9.6% dari negaranya. Keduanya adalah basis pelanggan luar negeri yang paling tahan terhadap hilangnya satu akun.
- Implikasi bisnis: kehilangan satu akun seperti 14646 atau 14911 akan memukul pendapatan Netherlands/EIRE secara drastis. Retensi dan hubungan akun kunci (*key account management*) di pasar ini lebih mendesak daripada di Germany/France.

**Catatan metodologis**
- Sesuai instruksi, hanya negara yang difilter (tanpa filter `Quantity`), sehingga total belanja adalah **neto setelah retur**. Penjualan kotor non-UK £1,603,016.35 dikurangi retur £70,823.93 sama dengan £1,532,192.42. Empat pelanggan non-UK bernilai neto negatif karena returnya melebihi pembeliannya.

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS A2: Analisis Potensi Kerugian Transaksi Pembatalan/Retur
# Instruksi:
# 1. Filter dataset untuk mencari transaksi pembatalan, yaitu yang InvoiceNo diawali huruf 'C' ATAU Quantity < 0.
# 2. Hitung berapa jumlah total transaksi yang dibatalkan.
# 3. Hitung total estimasi nilai kerugian (Total Nilai Transaksi yang dibatalkan) dalam GBP (£).
# ------------------------------------------------------------------------------
banner("TUGAS A2 — ANALISIS TRANSAKSI PEMBATALAN / RETUR")

# 1. Filter transaksi pembatalan: InvoiceNo diawali 'C' ATAU Quantity < 0
rdd_cancel = rdd.filter(lambda x: x[0].upper().startswith('C') or x[3] < 0).persist()

# 2. Jumlah transaksi dibatalkan: dihitung per baris item transaksi DAN per nomor invoice unik
total_cancel_rows = rdd_cancel.count()
total_cancel_invoices = rdd_cancel.map(lambda x: x[0]).distinct().count()

# 3. Estimasi nilai kerugian = jumlah |Quantity x UnitPrice| (Quantity pembatalan bernilai negatif,
#    sehingga nilai absolut dipakai agar kerugian dinyatakan sebagai angka positif)
total_loss = rdd_cancel.map(lambda x: abs(x[3] * x[4])).sum()

print(f"Jumlah baris item transaksi dibatalkan : {total_cancel_rows:,}")
print(f"Jumlah nomor invoice unik dibatalkan   : {total_cancel_invoices:,}")
print(f"Total estimasi nilai kerugian          : £ {total_loss:,.2f}")

_ = rdd_cancel.unpersist()

#### 🔎 Interpretasi Hasil — Tugas A2 (Transaksi Pembatalan / Retur)

**Temuan utama**
- Terdapat **8,905 baris item** pembatalan pada **3,654 invoice unik**, dengan estimasi nilai **£611,342.09**.
- Baris pembatalan hanya **2.19%** dari 406,829 baris bersih, tetapi invoice pembatalan adalah **16.47%** dari 22,190 invoice. Dengan kata lain **sekitar satu dari enam invoice adalah pembatalan**, namun ukurannya kecil: rata-rata 2.44 baris per invoice pembatalan, dibandingkan 18.3 baris per invoice secara umum. Pembatalan cenderung berupa **retur sebagian**, bukan pembatalan seluruh pesanan.
- Nilai £611,342.09 setara **6.86%** dari penjualan non-pembatalan (£8,911,407.90). Rata-rata nilai per invoice pembatalan £167.31.

**Makna**
- **Nilai kerugian sangat terkonsentrasi pada tiga baris**: C581484 (£168,469.60), C541433 (£77,183.60), dan C556445 (£38,970.00, baris "Manual"), yaitu pembatalan atas tiga pesanan besar pada interpretasi Tahap 5. Ketiganya £284,623.20 atau **46.6% dari seluruh nilai**. Tanpa ketiganya, nilai pembatalan tersisa £326,718.89.
- Pembatalan terjadi di semua pasar. UK memiliki 7,533 baris (84.6% dari seluruh baris pembatalan; tingkat pembatalan UK 2.08% dari baris UK). **Germany** memiliki 453 baris dengan tingkat pembatalan **4.77%** dari 9,495 barisnya, sekitar **2.2 kali** rata-rata keseluruhan (2.19%), sehingga layak ditelusuri penyebabnya (kualitas produk, ketidaksesuaian pengiriman, atau kebiasaan pelanggan).
- Kedua kriteria filter (`InvoiceNo` berawalan "C" atau `Quantity < 0`) menghasilkan himpunan yang **sama persis** pada data bersih: seluruh 8,905 baris berawalan "C" dan berkuantitas negatif. Baris berkuantitas negatif tanpa awalan "C" ada pada dataset mentah, tetapi tidak memiliki `CustomerID` sehingga sudah terbuang pada tahap pembersihan.

**Catatan**
- Nilai ini adalah **nilai kotor transaksi yang dibatalkan** (jumlah |Quantity × UnitPrice|), bukan kerugian laba. Barang yang diretur bisa dijual kembali, dan sebagian pembatalan hanyalah koreksi pesanan yang kemudian diganti invoice baru.

### KATEGORI B: OPTIMASI PERFORMA & BENCHMARK RDD (PERFORMANCE ENGINEERING)

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS B1: Eksperimen Pengaruh Partisi Terhadap Waktu Eksekusi
# Instruksi:
# 1. Gunakan fungsi .repartition() untuk mengubah RDD ke dalam 3 variasi jumlah partisi: 2, 8, dan 16.
# 2. Jalankan operasi agregasi reduceByKey() pada masing-masing variasi partisi tersebut.
# 3. Ukur dan cetak waktu eksekusi (runtime) masing-masing variasi menggunakan pustaka `time`.
# ------------------------------------------------------------------------------
banner("TUGAS B1 — PENGARUH JUMLAH PARTISI TERHADAP WAKTU EKSEKUSI reduceByKey")
from pyspark import StorageLevel

partition_variants = [2, 8, 16]
N_RUNS = 3          # tiap variasi diukur 3 kali agar tidak bergantung pada satu pengukuran

b1_results = []
for n_part in partition_variants:
    # (a) repartition + pembentukan Pair RDD (InvoiceNo, TotalSpent); dimaterialisasi lebih dulu
    #     agar waktu yang diukur pada tahap (b) murni waktu agregasi reduceByKey.
    t0 = time.time()
    rdd_p = (rdd.repartition(n_part)
                .map(lambda x: (x[0], x[3] * x[4]))
                .persist(StorageLevel.MEMORY_AND_DISK))
    n_records = rdd_p.count()
    t_prep = time.time() - t0

    # (b) reduceByKey (jumlah partisi hasil = jumlah partisi input) diukur N_RUNS kali
    runs, n_keys = [], None
    for _ in range(N_RUNS):
        t1 = time.time()
        n_keys = rdd_p.reduceByKey(lambda a, b: a + b).count()   # count() = aksi pemicu eksekusi
        runs.append(time.time() - t1)

    b1_results.append({"n_part": rdd_p.getNumPartitions(), "prep": t_prep, "runs": runs,
                       "avg": sum(runs) / len(runs), "min": min(runs), "keys": n_keys})
    _ = rdd_p.unpersist()

print(f"Core/thread tersedia (defaultParallelism): {sc.defaultParallelism}")
print(f"Jumlah ulangan pengukuran per variasi    : {N_RUNS}\n")
print(f"{'Partisi':>8} | {'repartition+cache (s)':>22} | {'reduceByKey rata-rata (s)':>26} | {'tercepat (s)':>13} | {'invoice unik':>12}")
print("-" * 95)
for r in b1_results:
    print(f"{r['n_part']:>8} | {r['prep']:>22.4f} | {r['avg']:>26.4f} | {r['min']:>13.4f} | {r['keys']:>12,}")

print("\nRincian waktu tiap ulangan reduceByKey (detik):")
for r in b1_results:
    print(f"  {r['n_part']:>2} partisi : " + ", ".join(f"{t:.4f}" for t in r["runs"]))

# Validasi: hasil agregasi harus identik untuk semua variasi partisi
assert len({r["keys"] for r in b1_results}) == 1, "Jumlah key hasil agregasi berbeda antar-partisi!"
fastest = min(b1_results, key=lambda r: r["avg"])
slowest = max(b1_results, key=lambda r: r["avg"])
print(f"\nKesimpulan (berdasarkan pengukuran di atas): variasi tercepat = {fastest['n_part']} partisi "
      f"({fastest['avg']:.4f} detik), terlambat = {slowest['n_part']} partisi ({slowest['avg']:.4f} detik).")
print("Validasi: jumlah invoice unik sama pada seluruh variasi partisi -> hasil agregasi konsisten.")

#### 🔎 Interpretasi Hasil — Tugas B1 (Pengaruh Jumlah Partisi terhadap Waktu `reduceByKey`)

**Temuan utama (Colab, `defaultParallelism` = 2 core, 406,829 record)**
- Rata-rata waktu `reduceByKey` dari 3 ulangan: **2 partisi = 1.1044 detik** (rentang 0.8261-1.3067), **8 partisi = 1.2953 detik** (1.2020-1.4202), **16 partisi = 2.9174 detik** (2.2623-3.8498).
- Dari 2 ke 8 partisi waktu naik **17.3%**, tetapi **rentang ulangannya saling tumpang tindih**, sehingga selisih keduanya belum meyakinkan secara statistik. **16 partisi konsisten paling lambat**: ulangan tercepatnya (2.2623 detik) bahkan lebih lambat daripada ulangan terlambat variasi lain (1.4202 detik); rata-ratanya **2.64 kali** variasi 2 partisi.
- Pada semua variasi hasil agregasi **identik: 22,190 invoice unik**. Jumlah partisi hanya memengaruhi kecepatan, bukan kebenaran hasil.

**Makna**
- Dengan 2 core, **2 partisi (= jumlah core)** sudah memanfaatkan seluruh paralelisme dengan biaya tambahan minimum: setiap core mengerjakan satu partisi. **8 partisi** (4 per core) menambah sedikit beban penjadwalan task, sedangkan **16 partisi** (8 per core) menambah beban yang jauh lebih besar. Pada `reduceByKey` jumlah blok shuffle mengikuti (partisi map × partisi reduce): 2 × 2 = 4 blok, 8 × 8 = 64 blok, dan 16 × 16 = 256 blok. Biaya mengelola ratusan blok kecil itu melampaui manfaat pembagian kerja pada data hanya 406,829 record.
- Prinsip umum: jumlah partisi sebaiknya **mendekati sampai beberapa kali jumlah core**. Terlalu sedikit membuat core menganggur, terlalu banyak menambah overhead. Pada data sekecil ini rentang itu sempit, sehingga 2 dan 8 partisi setara.

**Catatan**
- Waktu tahap persiapan (`repartition + cache`: 2.4065, 1.8854, 2.2397 detik) tidak naik secara teratur. Variasi 2 partisi yang dijalankan pertama kemungkinan menanggung biaya pemanasan (proses Python/JVM), bukan efek jumlah partisi. Salah satu ulangan 16 partisi (3.8498 detik) juga jauh lebih lambat daripada dua lainnya, menandakan waktu Colab berfluktuasi, sehingga pengukuran berulang memang diperlukan.
- Jumlah 22,190 invoice berbeda dari 16,646 pada Tahap 6 karena B1 memakai `rdd` (semua negara, termasuk invoice pembatalan), sedangkan Tahap 6 memakai `rdd_uk_valid`.

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS B2: Perbandingan Performa reduceByKey vs groupByKey
# Instruksi:
# 1. Buat Pair RDD dengan Key = Description (Nama Produk) dan Value = Quantity.
# 2. Hitung total kuantitas terjual per produk menggunakan `reduceByKey()`, catat waktu eksekusinya.
# 3. Hitung total kuantitas terjual per produk menggunakan `groupByKey()` + `mapValues(sum)`, catat waktu eksekusinya.
# 4. Tampilkan perbandingan selisih waktu eksekusi kedua fungsi tersebut.
# ------------------------------------------------------------------------------
banner("TUGAS B2 — PERBANDINGAN reduceByKey vs groupByKey")
from operator import add

# 1. Pair RDD (Description, Quantity); dimaterialisasi lebih dulu agar kedua metode
#    dibandingkan pada input yang sama dan waktu baca CSV tidak ikut terukur.
rdd_prod_pair = rdd.map(lambda x: (x[2], x[3])).persist(StorageLevel.MEMORY_AND_DISK)
n_pairs = rdd_prod_pair.count()

N_RUNS = 3
t_reduce, t_group = [], []
for _ in range(N_RUNS):                       # diselang-seling agar adil terhadap efek urutan
    # 2. reduceByKey
    t0 = time.time()
    n_reduce = rdd_prod_pair.reduceByKey(add).count()
    t_reduce.append(time.time() - t0)
    # 3. groupByKey + mapValues(sum)
    t0 = time.time()
    n_group = rdd_prod_pair.groupByKey().mapValues(sum).count()
    t_group.append(time.time() - t0)

# Validasi: kedua metode harus menghasilkan total kuantitas yang identik untuk setiap produk
res_reduce = rdd_prod_pair.reduceByKey(add).collectAsMap()
res_group = rdd_prod_pair.groupByKey().mapValues(sum).collectAsMap()
assert res_reduce == res_group, "Hasil reduceByKey dan groupByKey berbeda!"

avg_r, avg_g = sum(t_reduce) / N_RUNS, sum(t_group) / N_RUNS

# 4. Perbandingan
print(f"Jumlah pasangan (Description, Quantity) : {n_pairs:,}")
print(f"Jumlah produk unik                      : {n_reduce:,} (reduceByKey) | {n_group:,} (groupByKey)")
print(f"Ulangan pengukuran                      : {N_RUNS}x per metode\n")
print(f"{'Metode':<28} | {'Rata-rata (s)':>13} | {'Tercepat (s)':>12} | Rincian ulangan (s)")
print("-" * 95)
print(f"{'reduceByKey':<28} | {avg_r:>13.4f} | {min(t_reduce):>12.4f} | " + ", ".join(f"{t:.4f}" for t in t_reduce))
print(f"{'groupByKey + mapValues(sum)':<28} | {avg_g:>13.4f} | {min(t_group):>12.4f} | " + ", ".join(f"{t:.4f}" for t in t_group))

selisih = avg_g - avg_r
print(f"\nSelisih waktu (groupByKey - reduceByKey) : {selisih:+.4f} detik")
if avg_r > 0:
    print(f"Rasio waktu (groupByKey / reduceByKey)   : {avg_g / avg_r:.2f}x")
if selisih > 0:
    print("Kesimpulan (berdasarkan pengukuran di atas): reduceByKey lebih cepat daripada groupByKey.")
else:
    print("Kesimpulan (berdasarkan pengukuran di atas): pada data & lingkungan ini groupByKey tidak lebih lambat "
          "dari reduceByKey (mode local, data relatif kecil); keunggulan reduceByKey lebih terlihat pada data/cluster besar.")
print("Validasi: total kuantitas per produk identik pada kedua metode.")

_ = rdd_prod_pair.unpersist()

#### 🔎 Interpretasi Hasil — Tugas B2 (`reduceByKey` vs `groupByKey`)

**Temuan utama (Colab, 2 core)**
- Kedua metode menghasilkan total kuantitas yang **identik** untuk 3,896 produk (divalidasi dengan `assert`).
- Rata-rata waktu: `reduceByKey` **1.5949 detik**, `groupByKey + mapValues(sum)` **1.8775 detik**. Selisih +0.2827 detik, rasio 1.18×: `reduceByKey` **15.1% lebih cepat**. Arah selisihnya konsisten pada ketiga ulangan berpasangan (1.1515 vs 1.2046; 1.4012 vs 1.4274; 2.2319 vs 3.0006).

**Makna**
- Hasil ini **sejalan dengan teori**: `reduceByKey` menggabungkan nilai per partisi lebih dulu (*map-side combine*), sehingga paling banyak 8 partisi × 3,896 kunci = **31,168 record** yang di-shuffle (7.7% dari 406,829). `groupByKey` mengirim **seluruh 406,829 pasangan**, sekitar 13.1 kali lebih banyak, lalu baru dijumlahkan.
- Namun keunggulannya **tidak besar pada data ini**: pada dua ulangan pertama selisihnya kecil (ulangan tercepat 1.1515 vs 1.2046 detik, hanya 4.4%), dan selisih mencolok hanya muncul pada ulangan ketiga yang lambat untuk kedua metode (2.2319 vs 3.0006 detik). Dengan data sekecil ini dan shuffle yang hanya berupa baca-tulis disk lokal, biaya jaringan yang menjadi keunggulan utama `reduceByKey` belum berperan. Pada uji lain di lingkungan 1 core, arah selisihnya bahkan sempat berbalik, sehingga selisih sekecil ini sensitif terhadap lingkungan.
- Risiko `groupByKey` bukan hanya kecepatan, tetapi **memori**: seluruh nilai satu kunci dikumpulkan dalam satu daftar. Aman di sini (kunci terbesar hanya 2,070 nilai), tetapi bisa menyebabkan *out-of-memory* pada kunci yang sangat dominan pada data besar.

**Kesimpulan praktis**
- Untuk agregasi berupa penjumlahan, `reduceByKey` adalah pilihan yang lebih aman dan skalabel; keunggulannya akan makin besar pada data dan cluster yang lebih besar. Jangan mengandalkan selisih detik pada data sekecil ini sebagai bukti tunggal.

### KATEGORI C: PEMROSESAN STRING & WAKTU (TEXT & DATE MANIPULATION)

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS C1: Analisis Jam Sibuk Transaksi (Peak Transaction Hours)
# Instruksi:
# 1. Buat fungsi parser atau ekstraksi Lambda untuk mengambil bagian 'Jam' (Hour) dari kolom `InvoiceDate` (Format: "MM/DD/YYYY HH:MM").
# 2. Bentuk Pair RDD (Jam, 1) dan gunakan reduceByKey untuk menghitung total frekuensi transaksi per jam.
# 3. Urutkan hasil dari jam yang paling sibuk dan tampilkan Top 3 Jam Sibuk!
# ------------------------------------------------------------------------------
banner("TUGAS C1 — TOP 3 JAM SIBUK TRANSAKSI")

# Catatan: tuple pada `rdd` tidak memuat InvoiceDate dan `rdd` tidak boleh diubah. Karena itu dibuat RDD
# terpisah (rdd_dates) dari df_clean -- baris yang sama dengan `rdd` (pembersihan identik) -- yang hanya
# berisi InvoiceDate. `rdd` dan `rdd_uk_valid` tetap utuh.
rdd_dates = df_clean.select(F.col("InvoiceDate").cast("string").alias("InvoiceDate")) \
                    .rdd.map(lambda row: row["InvoiceDate"])

# 1. Parser jam: mendukung "MM/DD/YYYY HH:MM" (mis. "12/1/2010 8:26") maupun "YYYY-MM-DD HH:MM:SS"
def extract_hour(date_str):
    try:
        time_part = date_str.strip().replace("T", " ").split(" ")[1]   # ambil bagian waktu
        hour = int(time_part.split(":")[0])
        return hour if 0 <= hour <= 23 else None
    except (AttributeError, IndexError, ValueError):
        return None

# 2. Pair RDD (Jam, 1) + reduceByKey
rdd_hour = rdd_dates.map(extract_hour).filter(lambda h: h is not None)
rdd_hour_count = rdd_hour.map(lambda h: (h, 1)).reduceByKey(lambda a, b: a + b)

# 3. Urutkan dari yang tersibuk, tampilkan Top 3
sorted_hours = rdd_hour_count.sortBy(lambda x: (-x[1], x[0])).collect()

print(f"Jumlah baris tanggal berhasil di-parse : {sum(c for _, c in sorted_hours):,}")
print("\nContoh format InvoiceDate pada data    :", rdd_dates.take(2))
print("\n--- TOP 3 JAM SIBUK TRANSAKSI ---")
for rank, (hour, freq) in enumerate(sorted_hours[:3], start=1):
    print(f"{rank}. Jam {hour:02d}:00 - {hour:02d}:59 | Frekuensi Transaksi: {freq:,}")

print("\nDistribusi lengkap frekuensi per jam (urut jam):")
for hour, freq in sorted(sorted_hours):
    print(f"  Jam {hour:02d} : {freq:,}")

#### 🔎 Interpretasi Hasil — Tugas C1 (Jam Sibuk Transaksi)

**Temuan utama**
- Tiga jam tersibuk berurutan: **12:00-12:59 (73,342 baris, 18.03%)**, **13:00-13:59 (65,062; 15.99%)**, dan **14:00-14:59 (55,075; 13.54%)**. Ketiganya bersama-sama **47.56%** dari 406,829 baris, padahal hanya 3 dari 15 jam aktif.
- Aktivitas terekam hanya pada **pukul 06:00-20:59** (tidak ada transaksi malam hingga dini hari). Rentang 10:00-15:59 menampung **80.84%** transaksi, dan 09:00-16:59 menampung 92.51%. Sebelum pukul 09:00 hanya 2.27%, dan sejak 17:00 hanya 5.22%.
- Puncak (jam 12) sekitar **8.3 kali** jam 08 dan **2.7 kali** rata-rata per jam aktif (27,122 baris). Terdapat penurunan tajam dari jam 15 (46,220) ke jam 16 (24,997), yaitu turun 45.9% dalam satu jam.

**Makna**
- Pola ini memperlihatkan aktivitas pembelian yang berpusat pada **jam kerja dengan puncak di tengah hari**, konsisten dengan pelanggan bisnis atau pembelian pada jam operasional. Penurunan drastis setelah jam 15 menunjukkan bahwa sore hari relatif sepi.
- Implikasi operasional: staf layanan pelanggan, kapasitas server/sistem pemesanan, dan kesiapan gudang sebaiknya paling kuat pada **11:00-15:00**; jam 18:00 ke atas hampir tidak memerlukan sumber daya yang sama. Waktu terbaik untuk mengirim promosi atau notifikasi adalah sesaat sebelum puncak (sekitar 10:00-11:00).

**Catatan metodologis**
- Satuan yang dihitung adalah **baris item**, bukan invoice. Jika dihitung per invoice unik, jam 12 (3,597) dan jam 13 (3,043) tetap teratas, tetapi peringkat 3 menjadi jam 11 (2,770), bukan jam 14. Itu terjadi karena keranjang di jam siang lebih besar (jam 12: 20.4 baris/invoice, dibanding rata-rata 18.3). Kesimpulan "puncak di sekitar jam 12-13" kuat, tetapi urutan ketiga sensitif terhadap cara menghitung.
- Perhitungan memakai seluruh baris data bersih, termasuk baris pembatalan, sesuai instruksi (tuple `rdd` tidak memuat `InvoiceDate`, sehingga dipakai RDD tanggal terpisah dari `df_clean`).

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS C2: Pengelompokan & Rata-rata Harga Produk Spesifik
# Instruksi:
# 1. Filter produk pada `rdd_uk_valid` yang deskripsi produknya mengandung kata kunci "HEART" (case-insensitive).
# 2. Hitung Rata-Rata Harga Jual (`UnitPrice`) untuk setiap varian produk yang mengandung kata "HEART".
# 3. Tampilkan total varian produk yang ditemukan beserta contoh 5 produk dan rata-rata harganya!
# ------------------------------------------------------------------------------
banner("TUGAS C2 — VARIAN PRODUK 'HEART' DAN RATA-RATA HARGANYA")

# 1. Filter deskripsi yang mengandung "HEART" (case-insensitive)
rdd_heart = rdd_uk_valid.filter(lambda x: "HEART" in x[2].upper())

# 2. Rata-rata UnitPrice per varian produk (Description): kumpulkan (jumlah harga, banyak transaksi)
rdd_heart_avg = (rdd_heart.map(lambda x: (x[2], (x[4], 1)))
                          .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                          .mapValues(lambda v: (v[0] / v[1], v[1])))      # (rata-rata harga, jumlah transaksi)

# 3. Total varian + contoh 5 produk (dipilih 5 varian dengan transaksi terbanyak; seri diurutkan alfabet)
n_heart_records = rdd_heart.count()
n_heart_variants = rdd_heart_avg.count()
sample5 = rdd_heart_avg.takeOrdered(5, key=lambda kv: (-kv[1][1], kv[0]))

print(f"Jumlah record transaksi produk 'HEART' : {n_heart_records:,}")
print(f"Total varian produk 'HEART' ditemukan  : {n_heart_variants:,}")
print("\n--- CONTOH 5 PRODUK 'HEART' (TRANSAKSI TERBANYAK) ---")
for product, (avg_price, n_tx) in sample5:
    print(f"Produk: {product:<42} | Rata-rata Harga: £ {avg_price:,.2f} | Jumlah Transaksi: {n_tx:,}")

#### 🔎 Interpretasi Hasil — Tugas C2 (Produk Bertema "HEART")

**Temuan utama**
- Terdapat **297 varian produk** berkata kunci "HEART" dengan **34,705 transaksi**, yaitu **9.79%** dari 354,321 baris UK valid, sedangkan varian tersebut hanya **7.73%** dari 3,844 nama produk UK. Rata-rata transaksi per varian HEART adalah 116.9, lebih tinggi **1.27 kali** daripada rata-rata seluruh produk (92.2). Kontribusinya ke volume 9.31% dan ke pendapatan 9.75%.
- Harga rata-rata produk HEART £2.65 per baris, sekitar **10.7% lebih rendah** daripada seluruh produk UK (£2.96); mediannya £1.65 berbanding £1.95. Rata-rata harga per varian berkisar **£0.19 sampai £35.65**, dengan median varian £1.65. Dua varian termahal adalah *3 TIER SWEETHEART GARDEN SHELF* (£35.65) dan *FRENCH STYLE EMBOSSED HEART CABINET* (£21.09).
- Lima varian dengan transaksi terbanyak berharga sempit, **£1.64 sampai £2.98**: *WHITE HANGING HEART T-LIGHT HOLDER* (1,940 transaksi, £2.89), *HEART OF WICKER SMALL* (952; £1.64), *NATURAL SLATE HEART CHALKBOARD* (951; £2.98), *HEART OF WICKER LARGE* (809; £2.96), dan *RED HANGING HEART T-LIGHT HOLDER* (659; £2.95).

**Makna**
- Tema HEART **sedikit lebih laku daripada porsinya di katalog**: ia mengisi 7.7% varian tetapi 9.8% transaksi, dengan harga cenderung terjangkau, ciri khas produk hadiah/dekorasi berharga rendah.
- Permintaan dalam tema ini cukup merata: lima varian teratas hanya **15.3%** dari transaksi HEART, dan *WHITE HANGING HEART T-LIGHT HOLDER* sendiri 5.59%. Namun terdapat **ekor panjang**: 55 varian (18.5%) hanya memiliki 10 transaksi atau kurang. Varian jarang terjual ini dapat dikaji untuk dipertahankan atau dihentikan.
- Harga rata-rata di sini adalah rata-rata `UnitPrice` per transaksi (tidak dibobot kuantitas). Rata-rata sensitif terhadap varian berharga tinggi (maksimum £35.65, sekitar 20 kali median varian), sehingga median (£1.65) lebih mewakili harga "tipikal" produk HEART.

**Catatan**
- Pencarian berbasis *substring* sehingga kata seperti "SWEETHEART" (30 varian, mis. *3 TIER SWEETHEART GARDEN SHELF*) ikut terhitung walau bukan bermotif hati secara harfiah. Bila yang dimaksud adalah motif hati, kata kunci dapat dipersempit dengan batas kata.

### KATEGORI D: OPERASI MULTISET & SET OPERATIONS (ADVANCED RDD CONCEPTS)

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS D1: Eksklusivitas Pasar (Operasi Himpunan RDD `subtract`)
# Instruksi:
# 1. Dapatkan daftar Nama Produk unik (`distinct()`) yang pernah dibeli di negara 'Germany'.
# 2. Dapatkan daftar Nama Produk unik (`distinct()`) yang pernah dibeli di negara 'France'.
# 3. Gunakan fungsi himpunan RDD `.subtract()` untuk menemukan produk yang dibeli di 'Germany' TETAPI TIDAK PERNAH dibeli di 'France'.
# 4. Tampilkan jumlah produk eksklusif tersebut dan cetak 5 contoh produknya!
# ------------------------------------------------------------------------------
banner("TUGAS D1 — PRODUK EKSKLUSIF GERMANY (TIDAK PERNAH DIBELI DI FRANCE)")

# 1 & 2. Produk unik per negara (sesuai instruksi: berdasarkan Country, tanpa filter tambahan)
rdd_prod_germany = rdd.filter(lambda x: x[6] == 'Germany').map(lambda x: x[2]).distinct()
rdd_prod_france  = rdd.filter(lambda x: x[6] == 'France').map(lambda x: x[2]).distinct()

# 3. Himpunan selisih: Germany - France
rdd_exclusive_de = rdd_prod_germany.subtract(rdd_prod_france)

# 4. Jumlah & 5 contoh (diurutkan alfabet agar deterministik)
n_de = rdd_prod_germany.count()
n_fr = rdd_prod_france.count()
n_exclusive = rdd_exclusive_de.count()
n_common = rdd_prod_germany.intersection(rdd_prod_france).count()
sample_exclusive = rdd_exclusive_de.takeOrdered(5)

print(f"Jumlah produk unik di Germany                    : {n_de:,}")
print(f"Jumlah produk unik di France                     : {n_fr:,}")
print(f"Produk dibeli di KEDUANYA (irisan)               : {n_common:,}")
print(f"Produk EKSKLUSIF Germany (tidak pernah di France): {n_exclusive:,}")
print("\n--- 5 CONTOH PRODUK EKSKLUSIF GERMANY ---")
for i, p in enumerate(sample_exclusive, start=1):
    print(f"{i}. {p}")

# Validasi konsistensi himpunan: |Germany| = |eksklusif| + |irisan|
assert n_de == n_exclusive + n_common, "Validasi himpunan gagal!"
print("\nValidasi: |Germany| = |eksklusif| + |irisan| -> terpenuhi.")

#### 🔎 Interpretasi Hasil — Tugas D1 (Produk Eksklusif Germany terhadap France)

**Temuan utama**
- Germany membeli **1,703 produk unik**, France **1,545**. Yang dibeli di keduanya **1,185 produk**; sisanya **518 produk eksklusif Germany (30.4%)**. Sebaliknya ada 360 produk eksklusif France. Tingkat kemiripan katalog (irisan dibagi gabungan, *Jaccard*) **57.4%**, dan **76.7%** produk France juga dibeli di Germany.
- Validasi himpunan terpenuhi: 1,703 = 518 + 1,185.

**Makna: "eksklusif" di sini sebagian besar berarti "jarang dibeli"**
- Kedua pasar berukuran sebanding (Germany 9,495 baris/95 pelanggan; France 8,491 baris/87 pelanggan), sehingga selisih katalog tidak sepenuhnya akibat volume data, meski Germany sedikit lebih besar.
- Kelima ratus delapan belas produk eksklusif itu hanya menyumbang **1,079 baris (11.4% baris Germany)** dan **10.0% pendapatan Germany**. Sebanyak **396 dari 518 (76.4%)** hanya muncul 2 baris atau kurang di Germany. Produk yang hanya dibeli satu-dua kali bisa saja tidak muncul di France karena kebetulan, bukan karena preferensi pasar. Jadi hasil ini lebih tepat dibaca sebagai "ekor panjang katalog Germany" daripada "produk khas selera Jerman".
- Sebaliknya, irisan 1,185 produk mencakup **69.6% katalog Germany** dan **76.7% katalog France**, yang menunjukkan bahwa inti portofolio kedua negara serupa, sehingga strategi stok dan promosi dapat banyak dipakai bersama.

**Catatan**
- Lima contoh yang ditampilkan diurutkan **alfabetis** (angka lebih dulu, lalu huruf), sehingga tidak mewakili produk eksklusif yang paling penting. Nama produk juga ada yang diawali spasi (mis. " RED SPOT GIFT BAG LARGE"); setelah dinormalisasi (menghapus spasi tepi dan menyeragamkan huruf) jumlah eksklusif hanya berubah dari 518 menjadi 517, jadi kesimpulan ini kokoh terhadap masalah penulisan nama.

## 🧭 Ringkasan Insight Keseluruhan (Penugasan 1)

1. **Kualitas data:** 24.93% record tidak memiliki `CustomerID` dan terbuang; analisis hanya mewakili transaksi pelanggan terdaftar, dan pasar UK mendominasi (≈ 89% baris bersih).
2. **Distorsi pesanan batal:** tiga invoice teratas (581483, 541431, 556444; total £299,784.10) memiliki pasangan pembatalan. Mereka membuat Top 10 invoice (5.93% total) dan peringkat 1-2 produk terlaris tampak lebih besar dari kenyataan. Produk terlaris sebenarnya (neto) adalah *WORLD WAR 2 GLIDERS ASSTD DESIGNS* dan *JUMBO BAG RED RETROSPOT*.
3. **Konsentrasi pelanggan luar negeri:** 1.18% pelanggan (5 dari 422) menghasilkan 44.35% belanja non-UK. Netherlands, EIRE, Australia, dan Sweden sangat bergantung pada 1-2 akun, sedangkan Germany dan France berbasis luas.
4. **Pembatalan:** 16.47% invoice adalah pembatalan, tetapi berukuran kecil (retur sebagian). Nilainya £611,342.09 (6.86% dari penjualan non-pembatalan), 46.6% di antaranya dari tiga pembatalan besar. Tingkat pembatalan Germany (4.77%) sekitar 2.2 kali rata-rata.
5. **Pola waktu:** transaksi berpusat pada jam kerja dengan puncak 12:00-14:00 (47.56% baris), dan turun tajam setelah pukul 15:59.
6. **Tema produk & pasar:** produk bertema HEART sedikit lebih laku (9.79% transaksi dari 7.73% varian) dengan harga lebih murah; katalog Germany dan France beririsan 57.4%, dan produk "eksklusif" Germany mayoritas hanya produk yang jarang dibeli.
7. **Performa Spark:** pada data sekecil ini (406,829 record) dan 2 core Colab, partisi sebanyak jumlah core (2) paling efisien dan 16 partisi sekitar 2.6 kali lebih lambat; caching mempercepat aksi berulang sekitar 2 kali; `reduceByKey` sekitar 15% lebih cepat daripada `groupByKey` (dengan variasi antarulangan yang besar). Keunggulan `reduceByKey` (13.1 kali lebih sedikit data yang di-shuffle) akan makin terasa pada data dan cluster yang jauh lebih besar.

## 9. Penyimpanan Output ke File TXT
Sel berikut menulis seluruh hasil `print()` dari notebook ini (berurutan sesuai tahapan) ke `/content/output_1.txt` dan mengunduhnya.

In [ ]:
# ==============================================================================
# PENYIMPANAN OUTPUT KE FILE TXT
# ==============================================================================
OUTPUT_TXT = "/content/output_1.txt"

header = (
    "OUTPUT PENUGASAN 1 - PEMROSESAN TERDISTRIBUSI PYSPARK RDD (ONLINE RETAIL)\n"
    + "Waktu eksekusi   : " + RUN_STARTED.strftime("%Y-%m-%d %H:%M:%S") + "\n"
    + "Versi PySpark    : " + spark.version + "\n"
    + "Versi Python     : " + sys.version.split()[0] + "\n"
    + "Sumber dataset   : " + "Google Drive (" + DRIVE_FILE + ")" + "\n"
    + "=" * 78 + "\n"
)

with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(header + LOG_BUFFER.getvalue())

# Verifikasi bahwa file benar-benar terbentuk dan dapat dibuka kembali
assert os.path.exists(OUTPUT_TXT) and os.path.getsize(OUTPUT_TXT) > 0, "File output gagal dibuat!"
with open(OUTPUT_TXT, encoding="utf-8") as f:
    isi = f.read()
print(f"File output tersimpan: {OUTPUT_TXT}  ({os.path.getsize(OUTPUT_TXT):,} byte, {isi.count(chr(10)) + 1} baris)")

# Di Google Colab, unduh file ke komputer Anda (diabaikan bila bukan di Colab)
try:
    from google.colab import files
    files.download(OUTPUT_TXT)
except ImportError:
    pass